<a href="https://colab.research.google.com/github/ggirlrottingg/ml-internship-starter/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggirlrottingg/ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

one row represents performance data for a single page over a specific period.
we are using fact_content_daily_performance table, time window of March 2026

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: Columns used to make predictions (e.g., historical clicks, impressions).

Label: What we are predicting.

Context: Non-predictive metadata (e.g., client ID, dates).

Excluded: Any column we ignore (e.g., Excluded raw URLs because they are high-cardinality text).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 (Grain): Show a sample row proving your definition from Step 1.

Query 2 (Counts & Dates): Output the total row count and date range for 2026-03.

Query 3 (Availability): Filter data using WHERE availability IS TRUE and check how many rows remain.

In [ ]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
DATA_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet"


# Query 1: Show sample rows (Grain Check)
query_grain = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM '{DATA_PATH}'
WHERE strftime(report_date, '%Y-%m') = '2026-03'
LIMIT 5;
"""
print("Query 1: Sample Rows (Grain) ---")
display(con.execute(query_grain).df())


# Query 2: Row Count & Date Span for March 2026
query_counts = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM '{DATA_PATH}'
WHERE strftime(report_date, '%Y-%m') = '2026-03';
"""
print("\nQuery 2: Row Count & Date Span ---")
display(con.execute(query_counts).df())


# Query 3: Availability Filter
query_availability = f"""
SELECT
    COUNT(*) AS available_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM '{DATA_PATH}'
WHERE strftime(report_date, '%Y-%m') = '2026-03'
  AND gsc_data_available IS TRUE;
"""
print("\nQuery 3: Available Rows ---")
display(con.execute(query_availability).df())

--- Query 1: Sample Rows (Grain) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727



--- Query 2: Row Count & Date Span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31



--- Query 3: Available Rows ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows,min_date,max_date
0,3611061,2026-03-01,2026-03-31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


**Unbalanced Historical Depth:** Pages and clients have varying tracking tenure. Older domains have extensive historical data, whereas newly integrated client sites possess very short performance windows, leading to inconsistent baseline feature depth across rows.

**GSC-Only Early Rows:** Search Console tracking often precedes GA4 integration. As a result, earlier performance records may contain valid search metrics (`gsc_impressions`, `gsc_clicks`) while lacking corresponding analytics data (`ga4_pageviews`).

**Non-Causal & Observational Nature:** This dataset measures observed search impression and click behaviors; it cannot reflect off-serp conversions, brand equity changes, offline marketing impact, or user actions outside the search engine results page.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge

df_raw = con.execute("""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet'
WHERE strftime(report_date, '%Y-%m') = '2026-03'
  AND gsc_data_available IS TRUE
LIMIT 50000;
""").df()

# Filling missing values
df_raw = df_raw.fillna(0)

# Feature 1: Historical Impression Volume (Log scale)
df_raw['f_log_impressions'] = df_raw['gsc_impressions'].apply(lambda x: 0 if x <= 0 else np.log1p(x))

# Feature 2: Historical Average Position
df_raw['f_avg_position'] = df_raw['gsc_avg_position']

# Feature 3: Is Top 10 Ranked (Binary flag)
df_raw['f_is_top_10'] = (df_raw['gsc_avg_position'] <= 10).astype(int)

# Feature 4: Estimated Historical CTR
df_raw['f_historical_ctr'] = df_raw['gsc_clicks'] / (df_raw['gsc_impressions'] + 1)

# Feature 5: GA4 Pageviews indicator
df_raw['f_ga4_views'] = df_raw['ga4_pageviews']

# Define Target Label (Future Clicks)
y = df_raw['gsc_clicks']

# LEAKAGE EXPERIMENT ---
# Intentionally introduce target leakage column
df_raw['f_LEAKED_COLUMN'] = y * 1.0

# Evaluate model WITH leakage
features_with_leak = ['f_log_impressions', 'f_avg_position', 'f_is_top_10', 'f_historical_ctr', 'f_ga4_views', 'f_LEAKED_COLUMN']
model_leak = Ridge().fit(df_raw[features_with_leak], y)
score_leak = r2_score(y, model_leak.predict(df_raw[features_with_leak]))
print(f"R^2 Score WITH Leakage (Artificial): {score_leak:.4f}")

# Remove leakage column & keep honest model
features_honest = ['f_log_impressions', 'f_avg_position', 'f_is_top_10', 'f_historical_ctr', 'f_ga4_views']
model_honest = Ridge().fit(df_raw[features_honest], y)
score_honest = r2_score(y, model_honest.predict(df_raw[features_honest]))
print(f"R^2 Score WITHOUT Leakage (Honest): {score_honest:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

R^2 Score WITH Leakage (Artificial): 1.0000
R^2 Score WITHOUT Leakage (Honest): 0.2715


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.